# Refusal Direction vs. Assistant Axis — Crescendo Analysis

**Goal**: Investigate how the refusal direction and assistant axis relate geometrically, and how
both projections evolve across turns during multi-turn (Crescendo) jailbreak conversations.

**Structure**:
- **Cell 1–2**: Load pre-computed vectors for all available models
- **Cell 3**: Per-layer cosine similarity between refusal direction and assistant axis
- **Cell 4–5**: Load crescendo trajectory data and plot refusal projections across turns
- **Cell 6**: Assistant projection trajectory across turns
- **Cell 7**: 2D trajectory scatter (refusal vs assistant projections)
- **Cell 8**: Layer-resolved heatmap for a single conversation

## Cell 1: Setup

In [ ]:
import torch
import numpy as np
import json
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

PROJ_ROOT = Path(".").resolve().parent  # assumes notebook is in notebooks/
VECTORS_DIR = PROJ_ROOT / "vectors"
EXPERIMENTS_DIR = PROJ_ROOT / "experiments" / "crescendo_runs"

print(f"Project root: {PROJ_ROOT}")
print(f"Vectors dir:  {VECTORS_DIR}")
print(f"Experiments:  {EXPERIMENTS_DIR}")
print(f"Available model dirs: {[p.name for p in sorted(VECTORS_DIR.iterdir()) if p.is_dir()]}")

## Cell 2: Load vectors for all available models

Loop over `VECTORS_DIR` subdirectories. For each model that has vectors:
- Load `assistant_axis.pt` (dict with `"axis"` key, shape: `n_layers x d_model`)
- Load `refusal_mean_diffs.pt` (shape: `n_pos x n_layers x d_model`)
- Load `metadata.json` (`{"pos": int, "layer": int}`)
- Extract the refusal direction per layer at the selected position

In [ ]:
models = {}

for model_dir in sorted(VECTORS_DIR.iterdir()):
    if not model_dir.is_dir():
        continue

    # Check that all required files exist
    axis_path = model_dir / "assistant_axis.pt"
    diffs_path = model_dir / "refusal_mean_diffs.pt"
    meta_path = model_dir / "metadata.json"

    missing = [p.name for p in [axis_path, diffs_path, meta_path] if not p.exists()]
    if missing:
        print(f"  SKIP {model_dir.name}: missing {missing}")
        continue

    # Load assistant axis — may be a dict with "axis" key or a raw tensor
    axis_raw = torch.load(axis_path, map_location="cpu", weights_only=True)
    if isinstance(axis_raw, dict):
        assistant_axis = axis_raw["axis"]  # (n_layers, d_model)
    else:
        assistant_axis = axis_raw  # already a tensor

    # Load refusal mean diffs: (n_pos, n_layers, d_model)
    mean_diffs = torch.load(diffs_path, map_location="cpu", weights_only=True)

    # Load metadata
    with open(meta_path) as f:
        metadata = json.load(f)
    selected_pos = metadata["pos"]   # e.g. -2
    selected_layer = metadata["layer"]  # e.g. 15

    n_positions = mean_diffs.shape[0]
    n_layers = mean_diffs.shape[1]
    d_model = mean_diffs.shape[2]

    # Convert negative position index to tensor index
    # mean_diffs is indexed [0..n_pos-1] mapping to positions [-n_pos..-1]
    pos_tensor_idx = n_positions + selected_pos  # e.g. n_pos + (-2) = n_pos - 2

    # Extract refusal direction per layer at the selected position: (n_layers, d_model)
    refusal_per_layer = mean_diffs[pos_tensor_idx]  # (n_layers, d_model)

    # Also load the single best refusal direction if available
    single_refusal_path = model_dir / "refusal_direction.pt"
    if single_refusal_path.exists():
        refusal_direction = torch.load(single_refusal_path, map_location="cpu", weights_only=True)
    else:
        refusal_direction = refusal_per_layer[selected_layer]

    models[model_dir.name] = {
        "assistant_axis": assistant_axis.float(),      # (n_layers, d_model)
        "refusal_per_layer": refusal_per_layer.float(),  # (n_layers, d_model)
        "refusal_direction": refusal_direction.float(),  # (d_model,) best single direction
        "mean_diffs": mean_diffs.float(),               # (n_pos, n_layers, d_model)
        "metadata": metadata,
        "selected_pos": selected_pos,
        "selected_layer": selected_layer,
        "n_layers": n_layers,
        "d_model": d_model,
        "n_positions": n_positions,
    }
    print(f"  Loaded {model_dir.name}: n_layers={n_layers}, d_model={d_model}, "
          f"selected layer={selected_layer}, pos={selected_pos}")

print(f"\nTotal models loaded: {len(models)}")

## Cell 3: Per-layer cosine similarity between refusal direction and assistant axis

For each model, compute cosine similarity at every layer. Includes:
- One line per model
- Highlighted peak refusal layer and peak axis layer
- Random baseline (mean +/- 2 sigma of cosine similarity with random unit vectors)

In [ ]:
def cosine_sim_per_layer(refusal, axis):
    """Compute cosine similarity between refusal and axis at each layer.
    
    Args:
        refusal: (n_layers, d_model) refusal direction per layer
        axis:    (n_layers, d_model) assistant axis per layer
    Returns:
        numpy array of shape (n_layers,) with cosine similarities
    """
    # F.cosine_similarity along dim=-1
    cos = torch.nn.functional.cosine_similarity(refusal, axis, dim=-1)
    return cos.numpy()


def random_baseline_cosine(d_model, n_samples=10000):
    """Compute expected cosine similarity between random unit vectors in R^d.
    
    For random unit vectors in R^d, cos_sim ~ N(0, 1/d), so:
      mean ~ 0, std ~ 1/sqrt(d)
    We verify empirically.
    """
    rng = np.random.default_rng(42)
    v1 = rng.standard_normal((n_samples, d_model))
    v2 = rng.standard_normal((n_samples, d_model))
    v1 /= np.linalg.norm(v1, axis=1, keepdims=True)
    v2 /= np.linalg.norm(v2, axis=1, keepdims=True)
    cos_sims = np.sum(v1 * v2, axis=1)
    return cos_sims.mean(), cos_sims.std()


# --- Compute cosine similarities for each model ---
cos_sim_data = {}
for model_name, m in models.items():
    cos = cosine_sim_per_layer(m["refusal_per_layer"], m["assistant_axis"])
    rand_mean, rand_std = random_baseline_cosine(m["d_model"])
    
    # Find peaks
    peak_cos_layer = int(np.argmax(np.abs(cos)))
    
    # Find peak refusal norm layer (where refusal direction is strongest)
    refusal_norms = torch.norm(m["refusal_per_layer"], dim=-1).numpy()
    peak_refusal_layer = int(np.argmax(refusal_norms))
    
    # Find peak axis norm layer (where assistant axis is strongest)
    axis_norms = torch.norm(m["assistant_axis"], dim=-1).numpy()
    peak_axis_layer = int(np.argmax(axis_norms))
    
    cos_sim_data[model_name] = {
        "cos": cos,
        "rand_mean": rand_mean,
        "rand_std": rand_std,
        "peak_cos_layer": peak_cos_layer,
        "peak_refusal_layer": peak_refusal_layer,
        "peak_axis_layer": peak_axis_layer,
        "refusal_norms": refusal_norms,
        "axis_norms": axis_norms,
        "selected_layer": m["selected_layer"],
    }
    
    print(f"{model_name}:")
    print(f"  Random baseline: mean={rand_mean:.4f}, std={rand_std:.4f} (theory: 1/sqrt({m['d_model']})={1/np.sqrt(m['d_model']):.4f})")
    print(f"  Peak |cos_sim| at layer {peak_cos_layer}: {cos[peak_cos_layer]:.4f} "
          f"(z={abs(cos[peak_cos_layer] - rand_mean)/rand_std:.1f}sigma)")
    print(f"  At refusal-selected layer {m['selected_layer']}: {cos[m['selected_layer']]:.4f} "
          f"(z={abs(cos[m['selected_layer']] - rand_mean)/rand_std:.1f}sigma)")
    print(f"  Peak refusal norm: layer {peak_refusal_layer}, peak axis norm: layer {peak_axis_layer}")
    print()

# --- Plot ---
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.7, 0.3],
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=["Cosine Similarity: Refusal Direction vs Assistant Axis",
                    "Relative Norm (refusal=solid, axis=dashed)"],
)

colors = px.colors.qualitative.Set2
for i, (model_name, data) in enumerate(cos_sim_data.items()):
    color = colors[i % len(colors)]
    n_layers = len(data["cos"])
    layers = np.arange(n_layers)
    
    # --- Top panel: cosine similarity ---
    fig.add_trace(
        go.Scatter(
            x=layers, y=data["cos"],
            mode="lines+markers",
            marker=dict(size=4),
            line=dict(color=color, width=2),
            name=model_name,
            legendgroup=model_name,
        ),
        row=1, col=1,
    )
    
    # Mark the selected refusal layer
    fig.add_trace(
        go.Scatter(
            x=[data["selected_layer"]],
            y=[data["cos"][data["selected_layer"]]],
            mode="markers",
            marker=dict(size=12, symbol="star", color=color, line=dict(color="black", width=1)),
            name=f"{model_name} refusal layer",
            legendgroup=model_name,
            showlegend=True,
            hovertext=f"Refusal layer {data['selected_layer']}<br>cos={data['cos'][data['selected_layer']]:.4f}",
            hoverinfo="text",
        ),
        row=1, col=1,
    )
    
    # Mark the peak |cosine| layer
    fig.add_trace(
        go.Scatter(
            x=[data["peak_cos_layer"]],
            y=[data["cos"][data["peak_cos_layer"]]],
            mode="markers",
            marker=dict(size=12, symbol="diamond", color=color, line=dict(color="black", width=1)),
            name=f"{model_name} peak cos",
            legendgroup=model_name,
            showlegend=True,
            hovertext=f"Peak |cos| layer {data['peak_cos_layer']}<br>cos={data['cos'][data['peak_cos_layer']]:.4f}",
            hoverinfo="text",
        ),
        row=1, col=1,
    )
    
    # Mark the peak axis norm layer
    fig.add_trace(
        go.Scatter(
            x=[data["peak_axis_layer"]],
            y=[data["cos"][data["peak_axis_layer"]]],
            mode="markers",
            marker=dict(size=10, symbol="square", color=color, line=dict(color="black", width=1)),
            name=f"{model_name} peak axis norm",
            legendgroup=model_name,
            showlegend=True,
            hovertext=f"Peak axis norm layer {data['peak_axis_layer']}<br>cos={data['cos'][data['peak_axis_layer']]:.4f}",
            hoverinfo="text",
        ),
        row=1, col=1,
    )
    
    # Random baseline band (use the first model's d_model for the band)
    if i == 0:
        fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.5, row=1, col=1)
        fig.add_hrect(
            y0=data["rand_mean"] - 2 * data["rand_std"],
            y1=data["rand_mean"] + 2 * data["rand_std"],
            fillcolor="green", opacity=0.08,
            line_width=0,
            row=1, col=1,
            annotation_text="random 2sigma", annotation_position="top right",
        )
    
    # --- Bottom panel: relative norms ---
    ref_norms_rel = data["refusal_norms"] / data["refusal_norms"].max()
    ax_norms_rel = data["axis_norms"] / data["axis_norms"].max()
    
    fig.add_trace(
        go.Bar(
            x=layers, y=ref_norms_rel,
            marker_color=color, opacity=0.5,
            name=f"{model_name} refusal norm",
            legendgroup=model_name,
            showlegend=False,
        ),
        row=2, col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=layers, y=ax_norms_rel,
            mode="lines",
            line=dict(color=color, width=2, dash="dash"),
            name=f"{model_name} axis norm",
            legendgroup=model_name,
            showlegend=False,
        ),
        row=2, col=1,
    )

fig.update_xaxes(title_text="Layer", row=2, col=1)
fig.update_yaxes(title_text="Cosine Similarity", row=1, col=1)
fig.update_yaxes(title_text="Relative Norm", row=2, col=1)
fig.update_layout(
    height=650, width=950,
    legend=dict(font=dict(size=10)),
    title_text="Refusal Direction vs. Assistant Axis — Per-Layer Analysis",
)
fig.show()

## Cell 4: Load crescendo trajectory data

Load all conversation results from `experiments/crescendo_runs/`. For each run:
- Read `config.json` for metadata (model, objective, max_turns, etc.)
- Read all `conversation_*.json` files
- Extract per-turn `refusal_proj` and `assistant_proj` at key layers
- Group by success/failure and whether turns were backtracked

In [ ]:
conversations = []  # list of dicts, one per conversation

for run_dir in sorted(EXPERIMENTS_DIR.iterdir()):
    if not run_dir.is_dir():
        continue
    config_path = run_dir / "config.json"
    if not config_path.exists():
        continue

    with open(config_path) as f:
        config = json.load(f)

    # Determine model name from config (try multiple keys for compatibility)
    model_full = config.get("model") or config.get("victim_model", "unknown")
    # Normalize to match vectors directory naming (e.g. "Qwen/Qwen-1_8B-Chat" -> "qwen-1_8b-chat")
    model_alias = model_full.split("/")[-1].lower().replace("_", "_").replace("-", "-")
    # Map common patterns to the vectors directory names
    model_alias_map = {
        "qwen-1_8b-chat": "qwen-1_8b-chat",
        "qwen-1.8b-chat": "qwen-1_8b-chat",
    }
    model_key = model_alias_map.get(model_alias, model_alias)

    # Find all conversation JSON files
    conv_files = sorted(run_dir.glob("conversation_*.json"))
    for conv_path in conv_files:
        with open(conv_path) as f:
            conv = json.load(f)

        # Skip conversations with errors or no turns
        if conv.get("error") or not conv.get("turns"):
            continue

        turns_data = []
        for turn in conv["turns"]:
            turn_info = {
                "turn": turn["turn"],
                "was_refused": turn.get("was_refused", False),
                "was_backtracked": turn.get("was_backtracked", False),
                "objective_score": turn.get("objective_score"),
                # refusal_proj and assistant_proj are dicts: {layer_str: float}
                "refusal_proj": {int(k): v for k, v in turn.get("refusal_proj", {}).items()},
                "assistant_proj": {int(k): v for k, v in turn.get("assistant_proj", {}).items()},
                "cos_refusal": {int(k): v for k, v in turn.get("cos_refusal", {}).items()},
                "cos_assistant": {int(k): v for k, v in turn.get("cos_assistant", {}).items()},
            }
            turns_data.append(turn_info)

        conversations.append({
            "run_id": config.get("run_id", run_dir.name),
            "model": model_full,
            "model_key": model_key,
            "objective": conv.get("objective", ""),
            "success": conv.get("success", False),
            "final_score": conv.get("final_score"),
            "n_turns": conv.get("n_turns", len(turns_data)),
            "backtrack_count": conv.get("backtrack_count", 0),
            "turns": turns_data,
        })

print(f"Loaded {len(conversations)} conversations from {len(list(EXPERIMENTS_DIR.iterdir()))} run directories")
print(f"  Successful: {sum(1 for c in conversations if c['success'])}")
print(f"  Failed:     {sum(1 for c in conversations if not c['success'])}")
print(f"  Models:     {set(c['model_key'] for c in conversations)}")

# Determine key layers from the first model that has vectors
# We use the refusal-selected layer for refusal_proj analysis
# and the peak axis norm layer for assistant_proj analysis
if models:
    first_model = list(models.values())[0]
    REFUSAL_LAYER = first_model["selected_layer"]
    
    # Peak axis layer: where assistant axis norm is largest
    axis_norms = torch.norm(first_model["assistant_axis"], dim=-1).numpy()
    AXIS_LAYER = int(np.argmax(axis_norms))
    
    print(f"\nKey layers for trajectory analysis:")
    print(f"  Refusal projection layer: {REFUSAL_LAYER} (from metadata)")
    print(f"  Assistant projection layer: {AXIS_LAYER} (peak axis norm)")

## Cell 5: Trajectory plot — refusal projection vs turn number

Mean +/- SE of `refusal_proj` at the refusal peak layer across turns, with separate
lines for successful vs failed conversations, and markers for backtracked turns.

In [ ]:
def collect_projections_by_turn(convos, layer, proj_key="refusal_proj"):
    """Collect projection values at a given layer, grouped by turn number and category.
    
    Returns:
        dict: {category: {turn_num: [values]}}
        where category is one of "successful", "failed", "backtracked"
    """
    groups = {"successful": {}, "failed": {}, "backtracked": {}}
    
    for conv in convos:
        cat = "successful" if conv["success"] else "failed"
        for turn in conv["turns"]:
            t = turn["turn"]
            proj = turn.get(proj_key, {})
            if layer not in proj:
                continue
            val = proj[layer]
            
            # Always add to the main category
            groups[cat].setdefault(t, []).append(val)
            
            # Also track backtracked turns separately
            if turn.get("was_backtracked"):
                groups["backtracked"].setdefault(t, []).append(val)
    
    return groups


def compute_mean_se(groups):
    """Convert {turn: [values]} to sorted arrays of (turns, means, ses)."""
    result = {}
    for cat, by_turn in groups.items():
        if not by_turn:
            continue
        turns = sorted(by_turn.keys())
        means = [np.mean(by_turn[t]) for t in turns]
        ses = [np.std(by_turn[t]) / np.sqrt(len(by_turn[t])) if len(by_turn[t]) > 1 else 0
               for t in turns]
        counts = [len(by_turn[t]) for t in turns]
        result[cat] = {
            "turns": np.array(turns),
            "means": np.array(means),
            "ses": np.array(ses),
            "counts": np.array(counts),
        }
    return result


# --- Refusal projection trajectory ---
refusal_groups = collect_projections_by_turn(conversations, REFUSAL_LAYER, "refusal_proj")
refusal_stats = compute_mean_se(refusal_groups)

fig = go.Figure()

cat_styles = {
    "successful": dict(color="rgba(44, 160, 44, 0.9)", dash="solid", symbol="circle"),
    "failed":     dict(color="rgba(214, 39, 40, 0.9)", dash="solid", symbol="square"),
    "backtracked": dict(color="rgba(255, 127, 14, 0.8)", dash="dash", symbol="x"),
}

for cat, style in cat_styles.items():
    if cat not in refusal_stats:
        continue
    s = refusal_stats[cat]
    
    # Mean line with markers
    fig.add_trace(go.Scatter(
        x=s["turns"], y=s["means"],
        mode="lines+markers",
        marker=dict(size=8, symbol=style["symbol"]),
        line=dict(color=style["color"], width=2, dash=style["dash"]),
        name=f"{cat} (n={s['counts'].max()})",
        hovertemplate="Turn %{x}<br>Mean refusal_proj: %{y:.3f}<br>n=%{customdata}<extra></extra>",
        customdata=s["counts"],
    ))
    
    # Shaded SE band
    if np.any(s["ses"] > 0):
        fig.add_trace(go.Scatter(
            x=np.concatenate([s["turns"], s["turns"][::-1]]),
            y=np.concatenate([s["means"] + s["ses"], (s["means"] - s["ses"])[::-1]]),
            fill="toself",
            fillcolor=style["color"].replace("0.9", "0.15").replace("0.8", "0.15"),
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        ))

fig.update_layout(
    title=f"Refusal Projection vs Turn Number (layer {REFUSAL_LAYER})",
    xaxis_title="Turn Number",
    yaxis_title=f"Signed Refusal Projection (layer {REFUSAL_LAYER})",
    height=450, width=850,
    legend=dict(x=0.02, y=0.98),
    hovermode="x unified",
)
fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.4)
fig.show()

# Print summary
for cat, s in refusal_stats.items():
    if len(s["turns"]) == 0:
        continue
    print(f"{cat}: turns {s['turns'].min()}-{s['turns'].max()}, "
          f"mean refusal_proj range [{s['means'].min():.2f}, {s['means'].max():.2f}]")

## Cell 6: Trajectory plot — assistant projection vs turn number

Same analysis as above but for `assistant_proj` at the assistant peak layer.

In [ ]:
# --- Assistant projection trajectory ---
asst_groups = collect_projections_by_turn(conversations, AXIS_LAYER, "assistant_proj")
asst_stats = compute_mean_se(asst_groups)

fig = go.Figure()

for cat, style in cat_styles.items():
    if cat not in asst_stats:
        continue
    s = asst_stats[cat]
    
    # Mean line with markers
    fig.add_trace(go.Scatter(
        x=s["turns"], y=s["means"],
        mode="lines+markers",
        marker=dict(size=8, symbol=style["symbol"]),
        line=dict(color=style["color"], width=2, dash=style["dash"]),
        name=f"{cat} (n={s['counts'].max()})",
        hovertemplate="Turn %{x}<br>Mean assistant_proj: %{y:.3f}<br>n=%{customdata}<extra></extra>",
        customdata=s["counts"],
    ))
    
    # Shaded SE band
    if np.any(s["ses"] > 0):
        fig.add_trace(go.Scatter(
            x=np.concatenate([s["turns"], s["turns"][::-1]]),
            y=np.concatenate([s["means"] + s["ses"], (s["means"] - s["ses"])[::-1]]),
            fill="toself",
            fillcolor=style["color"].replace("0.9", "0.15").replace("0.8", "0.15"),
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        ))

fig.update_layout(
    title=f"Assistant Projection vs Turn Number (layer {AXIS_LAYER})",
    xaxis_title="Turn Number",
    yaxis_title=f"Signed Assistant Projection (layer {AXIS_LAYER})",
    height=450, width=850,
    legend=dict(x=0.02, y=0.98),
    hovermode="x unified",
)
fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.4)
fig.show()

# Print summary
for cat, s in asst_stats.items():
    if len(s["turns"]) == 0:
        continue
    print(f"{cat}: turns {s['turns'].min()}-{s['turns'].max()}, "
          f"mean assistant_proj range [{s['means'].min():.2f}, {s['means'].max():.2f}]")

## Cell 7: 2D trajectory scatter

Scatter plot: `refusal_proj` (x) vs `assistant_proj` (y) colored by turn number.
Shows trajectory arrows for individual conversations to visualize how the model's
internal state migrates through refusal-assistant space during a crescendo attack.

In [ ]:
# Build a flat table of (refusal_proj, assistant_proj, turn, conv_id, success, backtracked)
scatter_rows = []
for ci, conv in enumerate(conversations):
    for turn in conv["turns"]:
        rp = turn["refusal_proj"].get(REFUSAL_LAYER)
        ap = turn["assistant_proj"].get(AXIS_LAYER)
        if rp is None or ap is None:
            continue
        scatter_rows.append({
            "refusal_proj": rp,
            "assistant_proj": ap,
            "turn": turn["turn"],
            "conv_idx": ci,
            "run_id": conv["run_id"],
            "success": conv["success"],
            "backtracked": turn.get("was_backtracked", False),
            "was_refused": turn.get("was_refused", False),
            "objective_score": turn.get("objective_score"),
        })

if not scatter_rows:
    print("No trajectory data available (conversations may have errors).")
else:
    max_turn = max(r["turn"] for r in scatter_rows)
    conv_indices = sorted(set(r["conv_idx"] for r in scatter_rows))

    fig = go.Figure()

    # --- Collect per-conversation trajectories ---
    all_trajs = []  # list of (xs, ys, turns, success)
    for ci in conv_indices:
        pts = sorted([r for r in scatter_rows if r["conv_idx"] == ci], key=lambda r: r["turn"])
        if len(pts) < 2:
            continue
        xs = [p["refusal_proj"] for p in pts]
        ys = [p["assistant_proj"] for p in pts]
        ts = [p["turn"] for p in pts]
        all_trajs.append((xs, ys, ts, pts[0]["success"], pts[0]["run_id"]))

    # --- Individual trajectories as faint background lines ---
    for xs, ys, ts, success, run_id in all_trajs:
        fig.add_trace(go.Scatter(
            x=xs, y=ys,
            mode="lines",
            line=dict(color="lightgray", width=1),
            opacity=0.35,
            showlegend=False,
            hoverinfo="skip",
        ))

    # --- Mean trajectory with confidence band ---
    # Align by turn number, compute mean/SE
    turn_set = sorted(set(r["turn"] for r in scatter_rows))
    mean_rp, mean_ap, se_rp, se_ap = [], [], [], []
    valid_turns = []
    for t in turn_set:
        rps = [r["refusal_proj"] for r in scatter_rows if r["turn"] == t]
        aps = [r["assistant_proj"] for r in scatter_rows if r["turn"] == t]
        if len(rps) >= 2:
            valid_turns.append(t)
            mean_rp.append(np.mean(rps))
            mean_ap.append(np.mean(aps))
            se_rp.append(np.std(rps) / np.sqrt(len(rps)))
            se_ap.append(np.std(aps) / np.sqrt(len(aps)))

    mean_rp, mean_ap = np.array(mean_rp), np.array(mean_ap)
    se_rp, se_ap = np.array(se_rp), np.array(se_ap)

    # SE envelope (refusal ± SE, assistant ± SE)
    upper_rp = mean_rp + se_rp
    lower_rp = mean_rp - se_rp
    upper_ap = mean_ap + se_ap
    lower_ap = mean_ap - se_ap

    fig.add_trace(go.Scatter(
        x=np.concatenate([upper_rp, lower_rp[::-1]]),
        y=np.concatenate([upper_ap, lower_ap[::-1]]),
        fill="toself",
        fillcolor="rgba(99,110,250,0.12)",
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip",
    ))

    # Mean trajectory line — gradient via marker coloring
    fig.add_trace(go.Scatter(
        x=mean_rp, y=mean_ap,
        mode="lines+markers",
        line=dict(color="rgba(99,110,250,0.8)", width=3),
        marker=dict(
            size=10, color=valid_turns,
            colorscale="Viridis", showscale=True,
            colorbar=dict(title="Turn"),
            line=dict(width=1, color="white"),
        ),
        name="Mean trajectory",
        hovertemplate=(
            "Turn %{customdata}<br>"
            "refusal_proj: %{x:.3f}<br>"
            "assistant_proj: %{y:.3f}<extra></extra>"
        ),
        customdata=valid_turns,
    ))

    # Arrows at start → second point, and at second-to-last → last point
    if len(mean_rp) >= 2:
        for i, label in [(0, "Start"), (len(mean_rp) - 2, "")]:
            fig.add_annotation(
                x=mean_rp[i + 1], y=mean_ap[i + 1],
                ax=mean_rp[i], ay=mean_ap[i],
                xref="x", yref="y", axref="x", ayref="y",
                showarrow=True,
                arrowhead=3, arrowsize=1.5, arrowwidth=2.5,
                arrowcolor="rgba(99,110,250,0.9)",
            )
        # Label start and end
        fig.add_annotation(
            x=mean_rp[0], y=mean_ap[0],
            text=f"Turn {valid_turns[0]}", showarrow=False,
            font=dict(size=11, color="green"), xshift=-15, yshift=10,
        )
        fig.add_annotation(
            x=mean_rp[-1], y=mean_ap[-1],
            text=f"Turn {valid_turns[-1]}", showarrow=False,
            font=dict(size=11, color="red"), xshift=15, yshift=-10,
        )

    # --- Highlight backtracked / refused points ---
    backtracked_arr = np.array([r["backtracked"] for r in scatter_rows])
    refused_arr = np.array([r["was_refused"] for r in scatter_rows])
    rp_arr = np.array([r["refusal_proj"] for r in scatter_rows])
    ap_arr = np.array([r["assistant_proj"] for r in scatter_rows])
    turns_arr = np.array([r["turn"] for r in scatter_rows])

    mask_bt = backtracked_arr
    if mask_bt.any():
        fig.add_trace(go.Scatter(
            x=rp_arr[mask_bt], y=ap_arr[mask_bt],
            mode="markers",
            marker=dict(
                size=10, color="rgba(0,0,0,0)",
                symbol="x", line=dict(width=2, color="red"),
            ),
            name="Backtracked",
            hovertemplate=(
                "Turn %{customdata[0]} (BACKTRACKED)<br>"
                "refusal: %{x:.3f}, assistant: %{y:.3f}<br>"
                "run: %{customdata[1]}<extra></extra>"
            ),
            customdata=[(r["turn"], r["run_id"][-6:]) for r, m in zip(scatter_rows, mask_bt) if m],
        ))

    mask_ref = refused_arr & ~backtracked_arr
    if mask_ref.any():
        fig.add_trace(go.Scatter(
            x=rp_arr[mask_ref], y=ap_arr[mask_ref],
            mode="markers",
            marker=dict(
                size=10, color="rgba(0,0,0,0)",
                symbol="diamond", line=dict(width=2, color="orange"),
            ),
            name="Refused",
            hovertemplate=(
                "Turn %{customdata[0]} (REFUSED)<br>"
                "refusal: %{x:.3f}, assistant: %{y:.3f}<br>"
                "run: %{customdata[1]}<extra></extra>"
            ),
            customdata=[(r["turn"], r["run_id"][-6:]) for r, m in zip(scatter_rows, mask_ref) if m],
        ))

    fig.update_layout(
        title=f"2D Trajectory: Refusal vs Assistant Projection<br>"
              f"<sub>Mean ± SE across {len(all_trajs)} conversations | "
              f"refusal layer {REFUSAL_LAYER}, assistant layer {AXIS_LAYER}</sub>",
        xaxis_title=f"Refusal Projection (layer {REFUSAL_LAYER})",
        yaxis_title=f"Assistant Projection (layer {AXIS_LAYER})",
        height=600, width=800,
        legend=dict(x=1.02, y=1, font=dict(size=10)),
    )
    fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.3)
    fig.add_vline(x=0, line_dash="dot", line_color="gray", opacity=0.3)
    fig.show()

    print(f"Total points: {len(scatter_rows)} across {len(conv_indices)} conversations")

## Cell 8: Layer-resolved heatmap

For one example conversation, show a heatmap: X = turn number, Y = layer,
color = `assistant_proj`. This reveals which layers shift most during the attack
and at which turns the largest changes occur.

In [ ]:
# Pick the conversation with the most turns for the best visualization
# Prefer conversations with actual projection data
valid_convos = [c for c in conversations if c["turns"] and c["turns"][0].get("assistant_proj")]
if not valid_convos:
    print("No conversations with projection data found.")
else:
    # Sort by number of turns (descending), then by backtrack_count (descending, more interesting)
    example_conv = sorted(valid_convos, key=lambda c: (c["n_turns"], c["backtrack_count"]), reverse=True)[0]
    
    print(f"Example conversation: {example_conv['run_id']}")
    print(f"  Turns: {example_conv['n_turns']}, Backtracks: {example_conv['backtrack_count']}, "
          f"Success: {example_conv['success']}, Final score: {example_conv['final_score']}")
    
    # Determine the set of layers from the first turn's projection data
    all_layers = sorted(example_conv["turns"][0]["assistant_proj"].keys())
    n_layers_conv = len(all_layers)
    n_turns_conv = len(example_conv["turns"])
    
    # Build heatmap matrices: (n_layers, n_turns)
    asst_matrix = np.full((n_layers_conv, n_turns_conv), np.nan)
    refusal_matrix = np.full((n_layers_conv, n_turns_conv), np.nan)
    
    turn_labels = []
    for ti, turn in enumerate(example_conv["turns"]):
        t_num = turn["turn"]
        suffix = ""
        if turn.get("was_backtracked"):
            suffix = " (BT)"
        elif turn.get("was_refused"):
            suffix = " (R)"
        turn_labels.append(f"T{t_num}{suffix}")
        
        for li, layer in enumerate(all_layers):
            ap = turn["assistant_proj"].get(layer)
            rp = turn["refusal_proj"].get(layer)
            if ap is not None:
                asst_matrix[li, ti] = ap
            if rp is not None:
                refusal_matrix[li, ti] = rp
    
    # --- Heatmap: assistant_proj ---
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Assistant Projection by Layer and Turn",
                        "Refusal Projection by Layer and Turn"],
        horizontal_spacing=0.12,
    )
    
    # Assistant projection heatmap
    fig.add_trace(
        go.Heatmap(
            z=asst_matrix,
            x=turn_labels,
            y=[str(l) for l in all_layers],
            colorscale="RdBu_r",
            zmid=0,
            colorbar=dict(title="Asst proj", x=0.45, len=0.9),
            hovertemplate="Turn: %{x}<br>Layer: %{y}<br>assistant_proj: %{z:.2f}<extra></extra>",
        ),
        row=1, col=1,
    )
    
    # Refusal projection heatmap
    fig.add_trace(
        go.Heatmap(
            z=refusal_matrix,
            x=turn_labels,
            y=[str(l) for l in all_layers],
            colorscale="RdBu_r",
            zmid=0,
            colorbar=dict(title="Ref proj", x=1.0, len=0.9),
            hovertemplate="Turn: %{x}<br>Layer: %{y}<br>refusal_proj: %{z:.2f}<extra></extra>",
        ),
        row=1, col=2,
    )
    
    fig.update_yaxes(title_text="Layer", row=1, col=1)
    fig.update_yaxes(title_text="Layer", row=1, col=2)
    fig.update_xaxes(title_text="Turn", row=1, col=1)
    fig.update_xaxes(title_text="Turn", row=1, col=2)
    
    fig.update_layout(
        height=500, width=1100,
        title_text=f"Layer-Resolved Projections — Conversation {example_conv['run_id']}<br>"
                   f"<sub>Success={example_conv['success']}, "
                   f"Backtracks={example_conv['backtrack_count']}, "
                   f"Final score={example_conv['final_score']}</sub>",
    )
    fig.show()
    
    # --- Also show a single assistant_proj heatmap with annotations for backtracked/refused turns ---
    fig2 = go.Figure()
    fig2.add_trace(go.Heatmap(
        z=asst_matrix,
        x=turn_labels,
        y=[str(l) for l in all_layers],
        colorscale="RdBu_r",
        zmid=0,
        colorbar=dict(title="Assistant<br>projection"),
        hovertemplate="Turn: %{x}<br>Layer: %{y}<br>assistant_proj: %{z:.2f}<extra></extra>",
    ))
    
    # Annotate the refusal-selected layer and axis peak layer
    if models:
        first_model_name = list(models.keys())[0]
        fig2.add_hline(
            y=REFUSAL_LAYER, line_dash="dash", line_color="red", opacity=0.7,
            annotation_text=f"refusal layer ({REFUSAL_LAYER})",
            annotation_position="top right",
        )
        fig2.add_hline(
            y=AXIS_LAYER, line_dash="dash", line_color="blue", opacity=0.7,
            annotation_text=f"axis peak layer ({AXIS_LAYER})",
            annotation_position="top right",
        )
    
    fig2.update_layout(
        title=f"Assistant Projection Heatmap — {example_conv['run_id']}",
        xaxis_title="Turn",
        yaxis_title="Layer",
        height=450, width=700,
    )
    fig2.show()